# Práctico Semana 8 — Anotación funcional y análisis de enriquecimiento
### Bioinformática en el Diagnóstico Molecular

**Modalidad:** demostrativa, guiada y con respuestas  
**Datos:** GSE193336, RNA-seq de macrófagos humanos no estimulados y estimulados con LPS.

## Objetivo
Aplicar un análisis de sobrerrepresentación (**ORA**) con Gene Ontology — Biological Process (**GO-BP**) e interpretar **background, GeneRatio, BgRatio, Count y p-value ajustado**.

> **Pregunta biológica:** ¿Qué procesos biológicos están sobrerrepresentados entre los genes que aumentan su expresión después de la estimulación con LPS?

### ¿Cómo trabajaremos?
En cada etapa seguiremos la misma lógica:

**Antes de ejecutar → Ejecutamos → Observamos → Respondemos → Interpretamos**

> **Importante:** el objetivo de hoy no es aprender DESeq2. El análisis diferencial se ejecutará como una etapa de preparación para obtener una lista de genes real.


## 1. Contexto del experimento

GSE193336 contiene RNA-seq de macrófagos humanos derivados de monocitos de **4 donantes**. Para cada donante se estudió una condición no estimulada y una condición estimulada con **50 ng/mL de LPS durante 24 h**.

En este práctico trabajaremos con:

**Datos reales → genes analizados (background) → genes de interés → GO-BP → dotplot → interpretación**


### Comprobación rápida

**Pregunta:** ¿Cuál es la comparación experimental de este práctico?

**Respuesta:** compararemos macrófagos humanos **estimulados con LPS** frente a macrófagos **no estimulados**.

**Pregunta:** ¿Cuál será nuestra unidad de interpretación?

**Respuesta:** no intentaremos interpretar cada gen por separado. Buscaremos **procesos biológicos GO compartidos** por los genes que aumentan su expresión después del estímulo.


## 2. Preparar el entorno


In [2]:
# Instalación de paquetes (puede tardar algunos minutos la primera vez)
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")

paquetes_bioc <- c("DESeq2", "clusterProfiler", "org.Hs.eg.db", "enrichplot")
faltan_bioc <- paquetes_bioc[!sapply(paquetes_bioc, requireNamespace, quietly = TRUE)]
if (length(faltan_bioc) > 0) BiocManager::install(faltan_bioc, ask = FALSE, update = FALSE)

library(DESeq2)
library(clusterProfiler)
library(org.Hs.eg.db)
library(enrichplot)
library(ggplot2)


## 3. Descargar y preparar los datos reales

Ejecute la siguiente celda completa. Esta sección descarga los datos, realiza un filtrado básico, ejecuta el análisis diferencial y construye el **background**.

### Antes de ejecutar

**Pregunta 1:** ¿Qué representa el *background*?

**Piensa primero:** ¿qué genes tenían realmente la posibilidad de terminar en nuestra lista de interés?

**Pregunta 2:** ¿Por qué no corresponde simplemente a “todos los genes humanos”?

Ejecute ahora la celda y después revise la respuesta guiada.


In [ ]:
# Datos: GSE193336 (NCBI GEO)
url <- "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE193nnn/GSE193336/suppl/GSE193336_GEO_Processed_Data_Raw_Count_Macrophage.csv.gz"
archivo <- "GSE193336_counts.csv.gz"

if (!file.exists(archivo)) {
  download.file(url, archivo, mode = "wb")
}

datos <- read.csv(archivo, check.names = FALSE)
dim(datos)
head(datos[, 1:min(5, ncol(datos))])

# Identificar automáticamente las columnas de muestras.
# El experimento contiene 4 macrófagos no estimulados y 4 estimulados con LPS.
nombres <- colnames(datos)
cols_lps <- grep("LPS", nombres, ignore.case = TRUE, value = TRUE)
cols_ctrl <- grep("unstim|control|ctrl", nombres, ignore.case = TRUE, value = TRUE)

cat("Columnas LPS detectadas:\n"); print(cols_lps)
cat("\nColumnas control detectadas:\n"); print(cols_ctrl)

if (length(cols_lps) != 4 || length(cols_ctrl) != 4) {
  stop("No se detectaron automáticamente las 4 muestras LPS y 4 control. Revise colnames(datos).")
}

# La primera columna contiene el identificador del gen.
gene_id <- as.character(datos[[1]])
counts <- as.matrix(datos[, c(cols_ctrl, cols_lps)])
storage.mode(counts) <- "integer"
rownames(counts) <- gene_id

# Eliminar duplicados de identificadores, si existieran.
counts <- counts[!duplicated(rownames(counts)), , drop = FALSE]

condicion <- factor(c(rep("Control", length(cols_ctrl)), rep("LPS", length(cols_lps))),
                    levels = c("Control", "LPS"))
coldata <- data.frame(condicion = condicion, row.names = colnames(counts))

# Filtrado simple para definir el universo de genes analizados:
# conservar genes con al menos 10 lecturas totales.
keep <- rowSums(counts) >= 10
counts_f <- counts[keep, , drop = FALSE]

cat("Genes en el background después del filtrado:", nrow(counts_f), "\n")

# Análisis diferencial con DESeq2.
dds <- DESeqDataSetFromMatrix(countData = counts_f,
                              colData = coldata,
                              design = ~ condicion)
dds <- DESeq(dds, quiet = TRUE)
res <- results(dds, contrast = c("condicion", "LPS", "Control"))

DE <- as.data.frame(res)
DE$ENSEMBL <- rownames(DE)

# Quitar versión de Ensembl, si estuviera presente.
DE$ENSEMBL_limpio <- sub("\\..*$", "", DE$ENSEMBL)

# Anotar manteniendo una fila por gen
DE$SYMBOL <- mapIds(
    org.Hs.eg.db,
    keys = DE$ENSEMBL_limpio,
    column = "SYMBOL",
    keytype = "ENSEMBL",
    multiVals = "first"
)

DE$ENTREZID <- mapIds(
    org.Hs.eg.db,
    keys = DE$ENSEMBL_limpio,
    column = "ENTREZID",
    keytype = "ENSEMBL",
    multiVals = "first"
)

# Background: genes analizados que pudieron mapearse a ENTREZID
background_entrez <- unique(na.omit(DE$ENTREZID))

cat("Genes analizados después del filtrado:", nrow(DE), "\n")
cat("Genes del background con ENTREZID:", length(background_entrez), "\n")

### Respuesta — ¿Qué es el background?

El **background** o universo corresponde al conjunto de genes que **pudieron ser evaluados y eventualmente seleccionados** en nuestro experimento.

En este práctico se construye a partir de los genes que:

1. permanecieron después del filtrado de expresión;
2. fueron incluidos en el análisis diferencial; y
3. pudieron mapearse a un identificador ENTREZ.

**¿Por qué no usamos simplemente todos los genes humanos?**

Porque muchos genes del genoma pueden no haber sido detectados o analizados en este experimento. Incluirlos modificaría artificialmente la frecuencia esperada de los términos GO.

> **Idea clave:** el enriquecimiento compara nuestra lista de interés contra lo que era posible observar en **este experimento**.


## 4. Construir nuestra lista de interés

Para el práctico seleccionaremos genes **sobreexpresados** después de LPS utilizando:

- `padj < 0.05`
- `log2FoldChange > 1`

Antes de ejecutar: ¿qué significa cada criterio?


In [ ]:
DE_up <- subset(DE,
                !is.na(padj) &
                !is.na(ENTREZID) &
                padj < 0.05 &
                log2FoldChange > 1)

genes_up <- unique(DE_up$ENTREZID)

cat("Genes sobreexpresados seleccionados:", length(genes_up), "\n")
head(DE_up[, c("SYMBOL", "log2FoldChange", "padj", "ENTREZID")])


### Interpretación guiada de nuestra lista de interés

#### 1. ¿Cuántos genes forman nuestra lista?

La celda anterior imprime:

`Genes sobreexpresados seleccionados: ...`

Ese número corresponde a los genes que cumplen **simultáneamente** los criterios definidos para este práctico.

> **Respuesta:** nuestra lista de interés está formada por los genes que presentan `padj < 0.05` y `log2FoldChange > 1`.

---

#### 2. ¿Qué representa `padj < 0.05`?

`padj` es el **p-value ajustado por múltiples comparaciones** del análisis diferencial.

> **Respuesta:** seleccionamos genes cuya diferencia de expresión conserva evidencia estadística después de considerar que se evaluaron muchos genes simultáneamente.

---

#### 3. ¿Qué representa `log2FoldChange > 1`?

Un `log2FoldChange = 1` corresponde a:

**2¹ = 2**

> **Respuesta:** `log2FoldChange > 1` selecciona genes cuya expresión es **más de dos veces mayor** en LPS respecto del control, según la estimación del modelo.

---

#### 4. ¿Lista de interés y background son lo mismo?

**No.**

- **Lista de interés (`genes_up`)**: genes que cumplen nuestros criterios de selección.
- **Background (`background_entrez`)**: genes analizados que podían haber sido seleccionados.

> **Idea clave:** el ORA no pregunta simplemente “¿qué funciones aparecen?”. Pregunta si determinadas funciones aparecen **más de lo esperado en comparación con el background**.


## 5. Análisis de enriquecimiento GO — Biological Process

Ahora utilizaremos `enrichGO()` para realizar un análisis de sobrerrepresentación.

### Antes de ejecutar

Observe dos argumentos importantes:

- `gene = genes_up` → nuestra **lista de interés**.
- `universe = background_entrez` → nuestro **background**.

También utilizamos:

- `ont = "BP"` → analizaremos **Biological Process**.
- `pAdjustMethod = "BH"` → corrección de Benjamini-Hochberg.
- `readable = TRUE` → facilita la lectura de los genes en los resultados.

> **Pregunta:** ¿qué esperamos encontrar?  
> Procesos GO que estén representados en nuestra lista de genes sobreexpresados **más de lo esperado respecto del background**.


In [ ]:
ego_up <- enrichGO(
  gene          = genes_up,
  universe      = background_entrez,
  OrgDb         = org.Hs.eg.db,
  keyType       = "ENTREZID",
  ont           = "BP",
  pAdjustMethod = "BH",
  pvalueCutoff  = 0.05,
  qvalueCutoff  = 0.20,
  readable      = TRUE
)

resultado_GO <- as.data.frame(ego_up)
dim(resultado_GO)
head(resultado_GO[, c("ID", "Description", "GeneRatio",
                      "BgRatio", "pvalue", "p.adjust", "Count")], 10)


### Interpretación guiada de la tabla de enriquecimiento

La tabla contiene varios términos GO. Para comenzar, observaremos las primeras filas.

Puedes ejecutar adicionalmente:

```r
resultado_GO[1, c("Description", "GeneRatio", "BgRatio",
                  "Count", "pvalue", "p.adjust")]
```

#### ¿Cómo leemos cada columna?

**Description**  
Nombre del proceso biológico GO.

**GeneRatio**  
Proporción de genes de nuestra **lista de interés** asociados con ese término.

Por ejemplo, si apareciera `20/500`, significaría que 20 de los 500 genes considerados en la lista para el análisis están asociados con ese término.

**BgRatio**  
Proporción de genes del **background** asociados con el mismo término.

**Count**  
Número de genes de nuestra lista que contribuyen al término enriquecido.

**pvalue**  
Evidencia estadística antes de corregir por la gran cantidad de términos evaluados.

**p.adjust**  
P-value corregido por múltiples comparaciones. Es el valor que debemos priorizar para decidir qué términos conservan evidencia estadística después del ajuste.

---

### ¿Dónde está la idea de “enriquecimiento”?

No basta con observar un `Count` grande.

Debemos preguntarnos:

> **¿La proporción de genes asociados al término es mayor en nuestra lista de interés de lo que esperaríamos considerando el background?**

La prueba de sobrerrepresentación evalúa estadísticamente esa pregunta.

### Pregunta de comprobación

**¿El término con más genes (`Count`) tiene que ser necesariamente el más significativo?**

**Respuesta:** no. La significancia depende del solapamiento observado **en relación con el tamaño de la lista, el tamaño del término y su frecuencia en el background**, además del ajuste por múltiples comparaciones.


## 6. Visualizar los resultados


In [ ]:
dotplot(
    ego_up,
    showCategory = 12,
    font.size = 11
) +
  ggtitle("GO-BP: genes sobreexpresados después de LPS") +
  theme(
    plot.title = element_text(size = 14),
    axis.text.y = element_text(size = 10)
  )

### Lectura guiada del dotplot

Leamos el gráfico siempre en el mismo orden.

#### 1. ¿Qué representa el eje Y?

Los **términos/procesos biológicos GO** enriquecidos.

#### 2. ¿Qué representa el eje X (`GeneRatio`)?

La proporción de genes de nuestra lista asociados con cada término.

> Un punto más hacia la derecha indica un **GeneRatio mayor**.

#### 3. ¿Qué representa el tamaño de los puntos?

El número de genes de nuestra lista asociados con el término (`Count`).

> **Punto más grande ≠ proceso más importante.**

#### 4. ¿Qué representa el color?

La significancia estadística indicada por la escala del gráfico, normalmente basada en `p.adjust`.

> Siempre debemos mirar la **leyenda** antes de interpretar el color.

---

### Ahora interpretemos un término juntos

Seleccione uno de los procesos del eje Y y siga estas preguntas:

1. ¿Cuál es su `GeneRatio` aproximado?
2. ¿Su punto es grande o pequeño?
3. ¿Qué indica su color según la leyenda?
4. ¿Tiene sentido biológico encontrar ese proceso en macrófagos estimulados con LPS?

### Pregunta de comprobación

**¿El término con mayor GeneRatio tiene necesariamente el menor `p.adjust`?**

**Respuesta:** no.

`GeneRatio` y `p.adjust` describen aspectos diferentes. El primero resume la proporción de genes de nuestra lista asociados al término; el segundo resume la evidencia estadística después de considerar el background y las múltiples comparaciones.

> **Para interpretar un resultado no debemos mirar una sola característica del gráfico.**


## 7. De la estadística a la interpretación biológica

### Pregunta final

> **¿Podemos afirmar, solo a partir del enriquecimiento, que un proceso biológico está “activado”?**

### Respuesta guiada

**No.**

El ORA nos indica que genes asociados con un proceso están **sobrerrepresentados** en nuestra lista respecto del background.

En este práctico existe además una información importante: construimos deliberadamente la lista utilizando genes **sobreexpresados después de LPS**. Por eso conocemos la dirección del cambio de los genes seleccionados.

Sin embargo:

> **Enriquecido ≠ automáticamente activado**

Para afirmar activación funcional de una vía o proceso necesitaríamos integrar información adicional, por ejemplo dirección y comportamiento de sus componentes, evidencia experimental y contexto biológico.

---

## 8. Integramos todo el práctico

**Datos reales**  
↓  
**Genes analizados = background**  
↓  
**Genes sobreexpresados = lista de interés**  
↓  
**Anotaciones GO-BP**  
↓  
**ORA: ¿qué aparece más de lo esperado?**  
↓  
**p.adjust + GeneRatio + Count**  
↓  
**Dotplot**  
↓  
**Interpretación biológica**

### Antes de terminar, comprueba que puedes responder

1. **¿Qué diferencia existe entre lista de interés y background?**  
   La lista contiene los genes seleccionados; el background contiene los genes que podían haber sido seleccionados.

2. **¿Qué significa que un término GO esté enriquecido?**  
   Que está sobrerrepresentado en nuestra lista respecto de lo esperado según el background.

3. **¿Por qué miramos `p.adjust`?**  
   Porque evaluamos muchos términos GO simultáneamente y debemos controlar el aumento de falsos positivos.

4. **¿Qué representa GeneRatio?**  
   La proporción de genes de la lista asociados con un término.

5. **¿Podemos decir “esta vía está activada”?**  
   No basándonos únicamente en un ORA.

---

### Idea de cierre

> **El análisis bioinformático no termina cuando aparece el gráfico. Termina cuando somos capaces de interpretar el resultado sin afirmar más de lo que los datos permiten.**
